In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/dias-rapids-25.02/lib/python3.10/site-packages/IPython/core/extensions.py:145: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  mod.load_ipython_extension(self.shell)


In [3]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/rewritten/o4_mini_high/checkpoints/post_cell_4_try_1.pickle

trying: ['orig_output']
me:  8
trying: ['trip_data']


me:  9
trying: ['sns']
me:  0
trying: ['file_loc']
me:  1
trying: ['pd']
me:  0
trying: ['time']
me:  0
trying: ['plt']
me:  0


Declaring variable sns
Declaring variable pd
Declaring variable time
Declaring variable plt
Declaring variable file_loc
Declaring variable orig_output
Declaring variable trip_data


In [4]:
%%RecordEvent
%%time
### cell 5 ###

# compute duration in minutes on GPU by dividing the timedelta directly by a 1-minute Timedelta
dropoff = trip_data['tpep_dropoff_datetime']
pickup  = trip_data['tpep_pickup_datetime']

trip_data['duration'] = (dropoff - pickup) / pd.Timedelta(minutes=1)

# extract pickup/dropoff hours and day name (GPU)
trip_data['trip_pickup_hour']  = pickup.dt.hour
trip_data['trip_dropoff_hour'] = dropoff.dt.hour
trip_data['trip_day']          = pickup.dt.day_name()

# inspect	rip_data.info()
trip_data.head()

CPU times: user 101 ms, sys: 150 ms, total: 251 ms
Wall time: 254 ms


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,duration,trip_pickup_hour,trip_dropoff_hour,trip_day
0,2018-01-01 00:21:05,2018-01-01 00:24:23,1,0.5,41,24,2,4.5,0.5,0.5,0.00,0.0,0.3,5.80,3.300000,0,0,Monday
1,2018-01-01 00:44:55,2018-01-01 01:03:05,1,2.7,239,140,2,14.0,0.5,0.5,0.00,0.0,0.3,15.30,18.166667,0,1,Monday
2,2018-01-01 00:08:26,2018-01-01 00:14:21,2,0.8,262,141,1,6.0,0.5,0.5,1.00,0.0,0.3,8.30,5.916667,0,0,Monday
3,2018-01-01 00:20:22,2018-01-01 00:52:51,1,10.2,140,257,2,33.5,0.5,0.5,0.00,0.0,0.3,34.80,32.483333,0,0,Monday
4,2018-01-01 00:09:18,2018-01-01 00:27:06,2,2.5,246,239,1,12.5,0.5,0.5,2.75,0.0,0.3,16.55,17.800000,0,0,Monday


In [5]:
%Checkpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/rewritten/o4_mini_high/checkpoints/post_cell_5_try_3.pickle

migration speed (bps): 800162664.0769734
---------------------------
variables to migrate:
pickup 48
time 72
plt 72
orig_output 48
file_loc 113
pd 72
sns 72
dropoff 48
trip_data 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/new_notebooks/nyc-taxi/rewritten/o4_mini_high/checkpoints/post_cell_5_try_3.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['trip_data']
Intermediate variables ['file_loc']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 3 =======
Input variables []
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
  trip_data
    Input columns: set()
    Changed columns: set()
    Created columns: set()
    Deleted columns: {'store_and_fwd_flag', 'VendorID', 'RatecodeID'}
======= Cell 4 =======
Input variables []
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 5 =======
Input variables []
Active variables []
Intermediate variables ['dropoff', 'trip_data', 'pickup']
Future variabl

In [7]:
with open(
    "/home/dias-benchmarks/new_notebooks/nyc-taxi/opt_cell_exec_info_5_try_3.pkl", "wb"
) as f:
    pickle.dump(opt_cell_exec_info[5], f)

In [8]:
opt_output = Out.get(4)

In [9]:
trip_data_opt = trip_data
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/nyc-taxi/annotated/checkpoints/post_cell_5.pickle
assert compare_df(trip_data_opt, trip_data)

import numpy as np
from elastic.core.common.pandas import is_type_styler

is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(
    orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_opt_output_array = isinstance(
    opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray)
)
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif (
    (is_orig_output_pd or is_opt_output_pd)
    and (is_orig_output_array or is_opt_output_array)
) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output

trying: ['trip_data']


me:  11
trying: ['pd']
me:  0
trying: ['time']
me:  0
trying: ['sns']
me:  0
trying: ['orig_output']
me:  12
trying: ['file_loc']
me:  1
trying: ['plt']
me:  0


Declaring variable pd
Declaring variable time
Declaring variable sns
Declaring variable plt
Declaring variable file_loc
Declaring variable trip_data
Declaring variable orig_output
